# 5.3 · Multi round-trip requests (MRTR) and elicitation

**多轮往返请求（MRTR）与征询**

> **After this lesson you can:** Implement and drive the MRTR flow end to end — InputRequiredResult, inputResponses, protected requestState — and use form- and URL-mode elicitation safely, including confirmation gates for destructive tools.

Official reading: [Spec: Multi round-trip requests](https://modelcontextprotocol.io/specification/2026-07-28/basic/patterns/mrtr) · [Spec: Elicitation](https://modelcontextprotocol.io/specification/2026-07-28/client/elicitation)

Run the cells top to bottom (`Shift+Enter`), then change things and re-run. Everything runs locally: servers live in this kernel, in temp folders, or on localhost. Cells that call a real LLM are skipped unless `ANTHROPIC_API_KEY` is set.

### 1. A destructive tool with a confirmation gate (hand-rolled MRTR)
We write the flow explicitly so every field is visible. `delete_repository` never deletes on the first call. It:

- checks the client declared form elicitation (spec MUST; hand-rolled code must do this itself),
- returns an `InputRequiredResult` asking the user to **type the repository name**,
- on retry, acts only on `accept` with a matching name, re-asks on a typo, and reports `decline`/`cancel`.

`ctx.request_state` holds what we minted last round. The SDK encrypts it on the way out and verifies it on the way back.

In [ ]:
import json
import logging

import mcp_types as types
from mcp import Client
from mcp.server import MCPServer
from mcp.server.mcpserver import Context

logging.getLogger("mcp").setLevel(logging.CRITICAL)


def wire(model) -> str:
    return json.dumps(model.model_dump(by_alias=True, exclude_none=True, mode="json"), indent=2)


REPOS = {"acme/web", "acme/prod", "acme/docs"}
github = MCPServer("github-admin")


def confirm_request(repo: str, note: str = "") -> types.ElicitRequest:
    return types.ElicitRequest(params=types.ElicitRequestFormParams(
        message=f"{note}This permanently deletes {repo}. Type the full repository name to confirm.",
        requested_schema={"type": "object",
                          "properties": {"repo_name": {"type": "string", "title": "Repository name",
                                                       "minLength": 1, "maxLength": 100}},
                          "required": ["repo_name"]}))


@github.tool()
def delete_repository(repo: str, ctx: Context) -> str | types.InputRequiredResult:
    """Permanently delete a repository (asks the user to confirm first)."""
    if repo not in REPOS:
        return f"No such repository: {repo}"
    caps = ctx.client_capabilities
    if caps is None or caps.elicitation is None:
        return "Refused: deletion needs user confirmation, but this client cannot ask the user."

    answer = (ctx.input_responses or {}).get("confirm_delete")
    if answer is None:                                     # round 1, or the client sent nothing
        return types.InputRequiredResult(input_requests={"confirm_delete": confirm_request(repo)},
                                         request_state=json.dumps({"repo": repo, "round": 1}))
    state = json.loads(ctx.request_state)                  # already verified by the SDK
    if answer.action == "decline":
        return f"Not deleted: the user declined deleting {state['repo']}."
    if answer.action == "cancel":
        return "Not deleted: the confirmation was dismissed. Ask again later if needed."
    if (answer.content or {}).get("repo_name") != state["repo"]:
        return types.InputRequiredResult(              # spec: re-ask rather than error
            input_requests={"confirm_delete": confirm_request(repo, "The name did not match. ")},
            request_state=json.dumps({"repo": repo, "round": state["round"] + 1}))
    REPOS.discard(state["repo"])
    return f"Deleted {state['repo']} after {state['round']} confirmation round(s)."


print("tool registered; repos:", sorted(REPOS))

### 2. The automatic loop: `Client(elicitation_callback=...)`
The high-level `call_tool` sees `input_required`, routes each input request to our callback, and retries with `inputResponses` + `requestState` until the result is complete. Our scripted "user" plays four scenarios.

In [ ]:
answers = []


async def confirm_dialog(context, params):
    print(f"   [dialog from github-admin] {params.message}")
    action, typed = answers.pop(0)
    print(f"   [user] {action} {typed or ''}")
    return types.ElicitResult(action=action, content={"repo_name": typed} if typed else None)


scenarios = {"decline": [("decline", None)],
             "cancel": [("cancel", None)],
             "typo, then correct": [("accept", "acme/wbe"), ("accept", "acme/web")]}

async with Client(github, elicitation_callback=confirm_dialog) as client:
    for label, script in scenarios.items():
        answers[:] = script
        result = await client.call_tool("delete_repository", {"repo": "acme/web"})
        print(f"{label:>20} -> {result.content[0].text}\n")
print("repos now:", sorted(REPOS))

### 3. Drive it manually and read the raw JSON
`client.session.call_tool(..., allow_input_required=True)` returns the interim result instead of resolving it. This is how you'd persist a pending question (for example, a chat waiting for a user who went to lunch) and resume later. We then build `inputResponses` ourselves and retry with the **exact** `requestState`.

In [ ]:
REPOS.add("acme/web")

async with Client(github, elicitation_callback=confirm_dialog) as client:
    first = await client.session.call_tool("delete_repository", {"repo": "acme/docs"}, allow_input_required=True)
    print("ROUND 1 result:\n", wire(first))
    saved_state = first.request_state                      # opaque: store it, never parse it
    key = next(iter(first.input_requests))

    responses = {key: types.ElicitResult(action="accept", content={"repo_name": "acme/docs"})}
    final = await client.session.call_tool("delete_repository", {"repo": "acme/docs"},
                                           input_responses=responses, request_state=saved_state,
                                           allow_input_required=True)
    print("\nROUND 2 result:\n", wire(final))

The `requestState` we minted was `{"repo": "acme/docs", "round": 1}`, but the client only ever sees a `v1.…` token. The SDK's `RequestStateBoundary` sealed it with AES-256-GCM together with an expiry, the method, the tool name, an arguments digest and the server's name as audience.

### 4. Break it: tamper with and replay `requestState`
Treat the client as an attacker. We (a) flip characters in the token, (b) replay a valid token on a *different* repository, and (c) send state the server never minted. Each one is rejected before our handler runs, with the same error so the attacker learns nothing.

In [ ]:
from mcp.shared.exceptions import MCPError

async with Client(github, elicitation_callback=confirm_dialog) as client:
    first = await client.session.call_tool("delete_repository", {"repo": "acme/web"}, allow_input_required=True)
    good = first.request_state
    yes = {"confirm_delete": types.ElicitResult(action="accept", content={"repo_name": "acme/prod"})}
    attacks = {"bit-flipped token": (good[:-6] + ("A" * 6 if not good.endswith("AAAAAA") else "B" * 6), "acme/web"),
               "replayed on another repo": (good, "acme/prod"),
               "forged plaintext state": (json.dumps({"repo": "acme/prod", "round": 1}), "acme/prod")}
    for label, (state, repo) in attacks.items():
        try:
            await client.session.call_tool("delete_repository", {"repo": repo}, input_responses=yes,
                                           request_state=state, allow_input_required=True)
            print(f"{label:>26}: ACCEPTED (bad!)")
        except MCPError as err:
            print(f"{label:>26}: {err.error.code} {err.error.message} {err.error.data}")
print("acme/prod still exists:", "acme/prod" in REPOS)

### 5. On the wire: raw JSON-RPC over stdio
Now there's no SDK on the client side. Watch the **ids** (1 then 2), the `resultType` values, and the state being echoed byte-for-byte. The `_meta` envelope declares `elicitation: {form: {}}` on *every* request, which is per-request capability negotiation.

In [ ]:
import subprocess
import sys
import tempfile
import textwrap
from pathlib import Path

workdir = Path(tempfile.mkdtemp())
(workdir / "confirm_server.py").write_text(textwrap.dedent('''
    import json
    import mcp_types as types
    from mcp.server import MCPServer
    from mcp.server.mcpserver import Context

    server = MCPServer("confirm", log_level="CRITICAL")

    @server.tool()
    def drop_table(table: str, ctx: Context) -> str | types.InputRequiredResult:
        """Drop a database table after confirmation."""
        answer = (ctx.input_responses or {}).get("ok")
        if answer is None:
            return types.InputRequiredResult(
                input_requests={"ok": types.ElicitRequest(params=types.ElicitRequestFormParams(
                    message=f"Drop table {table}?",
                    requested_schema={"type": "object", "properties": {"confirm": {"type": "boolean"}},
                                      "required": ["confirm"]}))},
                request_state=json.dumps({"table": table}))
        ok = answer.action == "accept" and answer.content.get("confirm")
        return f"dropped {table}" if ok else f"kept {table}"

    if __name__ == "__main__":
        server.run(transport="stdio")
'''))

proc = subprocess.Popen([sys.executable, str(workdir / "confirm_server.py")],
                        stdin=subprocess.PIPE, stdout=subprocess.PIPE, text=True)
META = {"io.modelcontextprotocol/protocolVersion": "2026-07-28",
        "io.modelcontextprotocol/clientCapabilities": {"elicitation": {"form": {}}},
        "io.modelcontextprotocol/clientInfo": {"name": "by-hand", "version": "0.1"}}


def rpc(message):
    print(">>>", json.dumps(message)[:400])
    proc.stdin.write(json.dumps(message) + "\n")
    proc.stdin.flush()
    reply = json.loads(proc.stdout.readline())
    print("<<<", json.dumps(reply, indent=1)[:1100], "\n")
    return reply


call = {"name": "drop_table", "arguments": {"table": "users"}, "_meta": META}
r1 = rpc({"jsonrpc": "2.0", "id": 1, "method": "tools/call", "params": call})
r2 = rpc({"jsonrpc": "2.0", "id": 2, "method": "tools/call", "params": {
    **call, "inputResponses": {"ok": {"action": "accept", "content": {"confirm": True}}},
    "requestState": r1["result"]["requestState"]}})

Break it on the wire too: send a request that declares **no** capabilities. This server is hand-rolled and doesn't check, so it asks anyway. That violates the spec MUST, and it's why section 1's tool checks `ctx.client_capabilities`.

In [ ]:
no_caps = {**META, "io.modelcontextprotocol/clientCapabilities": {}}
r3 = rpc({"jsonrpc": "2.0", "id": 3, "method": "tools/call",
          "params": {"name": "drop_table", "arguments": {"table": "orders"}, "_meta": no_caps}})
print("server sent an elicitation to a client without the capability:", "inputRequests" in r3.get("result", {}))
proc.stdin.close()
exit_code = proc.wait(timeout=10)

### 6. Stateless scale-out: instance A asks, instance B finishes
Because the state travels with the request, the retry can land on a **different server instance**, provided they share the sealing key (and the same name, which is the token's audience). With the default per-process ephemeral key, instance C rejects A's token. This is the operational reason to configure `RequestStateSecurity(keys=[...])` behind a load balancer. A short TTL shows expiry too.

In [ ]:
import secrets
import time

from mcp.server.request_state import RequestStateSecurity

SHARED_KEY = secrets.token_hex(32)


def make_instance(label, security):
    srv = MCPServer("transfers", request_state_security=security)

    @srv.tool()
    def transfer(amount: int, ctx: Context) -> str | types.InputRequiredResult:
        """Transfer money after the user approves the amount."""
        answer = (ctx.input_responses or {}).get("approve")
        if answer is None:
            return types.InputRequiredResult(
                input_requests={"approve": types.ElicitRequest(params=types.ElicitRequestFormParams(
                    message=f"Approve transfer of {amount} EUR?",
                    requested_schema={"type": "object", "properties": {"approve": {"type": "boolean"}},
                                      "required": ["approve"]}))},
                request_state=json.dumps({"amount": amount}))
        return f"instance {label}: transferred {json.loads(ctx.request_state)['amount']} EUR"
    return srv


a = make_instance("A", RequestStateSecurity(keys=[SHARED_KEY], ttl=2))
b = make_instance("B", RequestStateSecurity(keys=[SHARED_KEY], ttl=2))
c = make_instance("C", None)                                          # default: ephemeral per-process key
approve = {"approve": types.ElicitResult(action="accept", content={"approve": True})}

async with Client(a, elicitation_callback=confirm_dialog) as ca, Client(b) as cb, Client(c) as cc:
    pending = await ca.session.call_tool("transfer", {"amount": 50}, allow_input_required=True)
    done = await cb.session.call_tool("transfer", {"amount": 50}, input_responses=approve,
                                      request_state=pending.request_state, allow_input_required=True)
    print("A asked, B finished ->", done.content[0].text)
    try:
        await cc.session.call_tool("transfer", {"amount": 50}, input_responses=approve,
                                   request_state=pending.request_state, allow_input_required=True)
    except MCPError as err:
        print("A asked, C (own ephemeral key) ->", err.error.message)
    time.sleep(2.2)
    try:
        await cb.session.call_tool("transfer", {"amount": 50}, input_responses=approve,
                                   request_state=pending.request_state, allow_input_required=True)
    except MCPError as err:
        print("after TTL, B ->", err.error.message)

The same valid token could be replayed on B *within* its TTL. A real transfer also needs a server-side "already used" record, since the spec says the protocol's measures don't guarantee single use.

### 7. URL mode: connect a third-party calendar without the secret touching the client
The server needs an OAuth grant for a third-party calendar. It sends a **URL-mode** elicitation and keeps a flow id in `requestState`. The client's callback shows the full URL and domain, gets consent and "opens a browser". Our fake browser completes the flow by calling the server's callback endpoint directly. The client only ever returns `{"action": "accept"}`. On retry, the server looks up the flow in **its own token store** (stateful, as the spec says for third-party tokens) and either finishes or asks again.

In [ ]:
import uuid
from urllib.parse import parse_qs, urlparse

calendar = MCPServer("calendar")
PENDING_FLOWS: dict[str, str] = {}      # flow_id -> user (server-side, bound to the authenticated user)
TOKENS: dict[str, str] = {}             # user -> third-party token (never sent to the client)


def oauth_callback(flow_id: str):
    """What the server's https://.../connect route would do after the third-party redirect."""
    user = PENDING_FLOWS.pop(flow_id)
    TOKENS[user] = f"tp-token-{secrets.token_hex(4)}"


@calendar.tool()
def next_meeting(ctx: Context) -> str | types.InputRequiredResult:
    """Return the user's next meeting from their third-party calendar."""
    user = "user-123"                                           # in production: the `sub` from MCP authorization
    if user in TOKENS:
        return f"Next meeting: Design review at 15:00 (fetched with a token the client never saw)"
    if ctx.request_state:                                       # retried, but the browser flow isn't done yet
        flow_id = json.loads(ctx.request_state)["flow"]
    else:
        flow_id = uuid.uuid4().hex[:8]
        PENDING_FLOWS[flow_id] = user
    return types.InputRequiredResult(
        input_requests={"connect": types.ElicitRequest(params=types.ElicitRequestURLParams(
            message="Connect your calendar so I can read your meetings.",
            url=f"https://mcp.example.com/connect?flow={flow_id}"))},
        request_state=json.dumps({"flow": flow_id}))


async def url_consent(context, params):
    if params.mode != "url":
        return types.ElicitResult(action="decline")
    domain = urlparse(params.url).hostname
    print(f"   [consent] calendar server wants to open: {params.url}\n   [consent] domain: {domain} -> user approves")
    flow = parse_qs(urlparse(params.url).query)["flow"][0]
    oauth_callback(flow)                                         # stands in for the user finishing OAuth in a browser
    return types.ElicitResult(action="accept")                   # no content in URL mode

async with Client(calendar, elicitation_callback=url_consent) as client:
    first = await client.session.call_tool("next_meeting", {}, allow_input_required=True)
    print("input request:", wire(first.input_requests["connect"]))
    print("\nauto loop ->", (await client.call_tool("next_meeting", {})).content[0].text)
    print("client-visible data contained a token?", any("tp-token" in json.dumps(v) for v in [wire(first)]))

There's no `elicitationId` and no `notifications/elicitation/complete`. The retry *is* the completion signal.

### 8. The declarative way: resolvers do the bookkeeping
For most tools you don't need hand-rolled state. A `Resolve(...)` resolver returns `Elicit(message, Model)`. The SDK builds the `InputRequiredResult`, records the answer in `requestState` so each question is asked only once, checks the client's capability (`-32021` if missing) and runs the body only when everything is resolved. Two resolvers here mean **two input requests in one round**.

In [ ]:
from typing import Annotated

from pydantic import BaseModel, Field

from mcp.server.elicitation import AcceptedElicitation, ElicitationResult
from mcp.server.mcpserver import Elicit, Resolve


class Environment(BaseModel):
    env: str = Field(json_schema_extra={"enum": ["staging", "production"]}, description="Target environment")


class Approval(BaseModel):
    approve: bool = Field(description="Approve this deployment?")


deployer = MCPServer("deployer")


def pick_env() -> Elicit[Environment]:
    return Elicit("Which environment?", Environment)


def approve(service: str) -> Elicit[Approval]:
    return Elicit(f"Deploy {service}? This restarts it.", Approval)


@deployer.tool()
def deploy(service: str,
           target: Annotated[Environment, Resolve(pick_env)],
           ok: Annotated[ElicitationResult[Approval], Resolve(approve)]) -> str:
    """Deploy a service after asking for the environment and an approval."""
    if isinstance(ok, AcceptedElicitation) and ok.data.approve:
        return f"Deployed {service} to {target.env}."
    return f"Deployment of {service} not approved ({ok.action})."


async def deploy_ui(context, params):
    props = params.requested_schema["properties"]
    content = {"env": "staging"} if "env" in props else {"approve": True}
    print(f"   [dialog] {params.message} -> {content}")
    return types.ElicitResult(action="accept", content=content)

async with Client(deployer, elicitation_callback=deploy_ui) as client:
    first = await client.session.call_tool("deploy", {"service": "api"}, allow_input_required=True)
    print("one round, input request keys:", list(first.input_requests))
    print((await client.call_tool("deploy", {"service": "api"})).content[0].text)

async with Client(deployer) as no_ui:
    try:
        await no_ui.call_tool("deploy", {"service": "api"})
    except MCPError as err:
        print("client without elicitation ->", err.error.code, err.error.message[:70])

### Your turn
1. Write a client-side MRTR driver yourself: a `while isinstance(result, types.InputRequiredResult)` loop around `client.session.call_tool(..., allow_input_required=True)` that dispatches each `inputRequests` entry to your own function, caps rounds at 5, and backs off when there are no `inputRequests`. Test it on `delete_repository` with the typo scenario.
2. Make section 1's `delete_repository` **single-use safe**: record each `requestState` it completes in a `USED` set (keyed by a hash of the token) and refuse a second completion. Show that replaying round 2 is now refused even within the TTL.
3. Turn the `Environment` schema in section 8 into a titled single-select enum using `oneOf: [{"const": "stg", "title": "Staging"}, ...]` in a hand-rolled request, and render it in your callback with the titles.
4. Design review: a teammate's server asks for the user's AWS secret key through a form-mode elicitation "because it's faster". List the spec rules it breaks and redesign it with URL mode, including how you'd defend against the phishing attack.